In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
data_dir = Path(r'C:\Users\DELL\Desktop\Pandas_in_14_days\Work\Data')
tables = {f.stem: pd.read_csv(f) for f in data_dir.glob('*.csv')}

for key, value in tables.items():
    print(key, value.shape)

olist_customers_dataset (99441, 5)
olist_geolocation_dataset (1000163, 5)
olist_orders_dataset (99441, 8)
olist_order_items_dataset (112650, 7)
olist_order_payments_dataset (103886, 5)
olist_order_reviews_dataset (99224, 7)
olist_products_dataset (32951, 9)
olist_sellers_dataset (3095, 4)
product_category_name_translation (71, 2)


In [3]:
customers = tables['olist_customers_dataset'].copy()
geolocation = tables['olist_geolocation_dataset'].copy()
orders = tables['olist_orders_dataset'].copy()
items = tables['olist_order_items_dataset'].copy()
payments = tables['olist_order_payments_dataset'].copy()
reviews = tables['olist_order_reviews_dataset'].copy()
products = tables['olist_products_dataset'].copy()
sellers = tables['olist_sellers_dataset'].copy()
category_translation = tables['product_category_name_translation'].copy()

df.groupby('column_to_group_by')['column_to_calculate'].aggregation_function()


Common Use Cases

• Single-Column Aggregation:
Group by one column and apply a summary function like sum(), mean(), count(), min(), or max().
#Total sales per category
df.groupby('category')['sales'].sum()

• Multi-Column Grouping:
Pass a list of columns to group by multiple criteria. Setting as_index=False keeps the grouped columns as regular DataFrame columns instead of turning them into a MultiIndex.

df.groupby(['region', 'category'], as_index=False)['sales'].mean()

• Multiple Aggregations at Once:
Use .agg() (or Named Aggregation) to compute several metrics simultaneously and rename the output columns cleanly.

df.groupby('category', as_index=False).agg(
    total_sales=('sales', 'sum'),
    avg_price=('price', 'mean'),
    order_count=('order_id', 'count')
)

• Group-Level Transformations:
Use .transform() when you want to return a result with the exact same shape and index as your original DataFrame—useful for calculating group percentages or filling missing values with a group average.

#Add a column showing each row's share of its category's total sales
df['pct_of_category'] = df['sales'] / df.groupby('category')['sales'].transform('sum')

• Filtering Groups:
Use .filter() to keep or drop entire groups based on a group-level condition.python
#Keep only categories with more than 100 total sales
df.groupby('category').filter(lambda x: x['sales'].sum() > 100)


### Helpful Parameters

• as_index=False. Mimics SQL-style GROUP BY output by preventing the grouping keys from becoming the DataFrame index.

• dropna=False. Includes NaN / null values as their own group instead of excluding them by default.

• sort=False. Disables automatic sorting of group keys to speed up performance on large datasets

## Task 1 - groupby with one and several keys, then agg with named aggregation

### 1.1 In payments, find the total payment_value for each payment_type, sorted from highest to lowest. Which type brings in the most money?

In [4]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [5]:
payments.groupby('payment_type')['payment_value'].sum().sort_values(ascending=False)

payment_type
credit_card    12542084.19
boleto          2869361.27
voucher          379436.87
debit_card       217989.79
not_defined           0.00
Name: payment_value, dtype: float64

### 1.2 In items, for each product_id, calculate the number of rows, the total price and the average price in one agg call with named aggregation. Show the 5 products with the highest total.

In [6]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [7]:
#Method 1
items.groupby('product_id', as_index=False).agg(
    num_of_rows = ('product_id','count'),
    total_price = ('price','sum'),
    avg_price = ('price','mean')
).sort_values(by='total_price', ascending=False).head()

,product_id,num_of_rows,total_price,avg_price
24086,bb50f2e236e5eea0100680137654686c,195,63885.00,327.615385
14068,6cdd53843498f92890544667809f1595,156,54730.20,350.834615
27613,d6160fb7873f184099d9bc95e30376af,35,48899.34,1397.124000
27039,d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51,137.651633
19742,99a4788cb24856965c36a24e339b6058,488,43025.56,88.167131


In [8]:
#Method 2
items.groupby('product_id').agg(
    n=('price', 'size'), 
    total=('price', 'sum'), 
    avg=('price', 'mean')).nlargest(5, 'total')

,n,total,avg
product_id,,,
bb50f2e236e5eea0100680137654686c,195,63885.00,327.615385
6cdd53843498f92890544667809f1595,156,54730.20,350.834615
d6160fb7873f184099d9bc95e30376af,35,48899.34,1397.124000
d1c427060a0f73f6b889a5c7c61f2ac4,343,47214.51,137.651633
99a4788cb24856965c36a24e339b6058,488,43025.56,88.167131


### 1.3 In customers, count customers for each (customer_state, customer_city) pair and show the top 10 pairs.

In [9]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [10]:
#Method 1
customers.groupby(['customer_state','customer_city'], as_index=False)['customer_unique_id'].size().nlargest(10,'size')

,customer_state,customer_city,size
4176,SP,sao paulo,15540
2788,RJ,rio de janeiro,6882
1062,MG,belo horizonte,2773
601,DF,brasilia,2131
2406,PR,curitiba,1521
3729,SP,campinas,1444
3208,RS,porto alegre,1379
372,BA,salvador,1245
3836,SP,guarulhos,1189
4160,SP,sao bernardo do campo,938


In [11]:
#Method 2
customers.groupby(['customer_state', 'customer_city']).size().sort_values(ascending=False).head(10)

customer_state  customer_city        
SP              sao paulo                15540
RJ              rio de janeiro            6882
MG              belo horizonte            2773
DF              brasilia                  2131
PR              curitiba                  1521
SP              campinas                  1444
RS              porto alegre              1379
BA              salvador                  1245
SP              guarulhos                 1189
                sao bernardo do campo      938
dtype: int64

### 1.4 In payments, calculate for each order_id the total payment_value and the number of payment rows. How many orders have more than one payment row?

In [12]:
payments.head(3)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


In [13]:
result = payments.groupby('order_id').agg(
    total_value = ('payment_value','sum'),
    num_of_rows = ('order_id','size')
)
(result.num_of_rows > 1).sum()

np.int64(2961)

## Use value_counts (with normalize=True), nunique, size vs count

### 2.1 What percentage of orders has each order_status? Use value_counts(normalize=True) and round to 2 decimals. Which status is the largest after 'delivered'?

In [14]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [15]:
orders['order_status'].value_counts(normalize=True).mul(100).round(2)

order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64

### 2.2 In customers, compare nunique() of customer_id and customer_unique_id. What does the gap mean?

In [16]:
customers.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


In [17]:
customers.customer_id.nunique()

99441

In [18]:
customers.customer_unique_id.nunique()

96096

In [19]:
customers.shape

(99441, 5)

customer_id is created for every order, while customer_unique_id identifies the person. The gap means some people ordered more than once.

### 2.3 In reviews, group by review_score and calculate both size() and count() of review_comment_message. Why are the numbers different?

In [20]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [21]:
reviews.groupby('review_score').agg(
    size = ('review_comment_message','size'),
    count = ('review_comment_message','count')
)

,size,count
review_score,,
1,11424,8745
2,3151,2145
3,8179,3557
4,19142,5976
5,57328,20554


size() counts rows, and count() counts only non-null values. The difference is the number of reviews with no comment at each score.

## Task 3 - Use reset_index and know why you need it

### 3.1 Turn the result of Q1.1 into a DataFrame with columns payment_type and total_payment, using reset_index. Confirm the type is DataFrame.

In [22]:
payments.groupby('payment_type')['payment_value'].sum().name

'payment_value'

In [23]:
df = payments.groupby('payment_type')['payment_value'].sum().reset_index(name='total_payment')

In [24]:
type(df)

pandas.DataFrame

In [25]:
df

,payment_type,total_payment
0,boleto,2869361.27
1,credit_card,12542084.19
2,debit_card,217989.79
3,not_defined,0.00
4,voucher,379436.87


### 3.2 Get the same table without reset_index, using as_index=False in the groupby. Do both give identical tables?

In [26]:
payments.groupby('payment_type', as_index=False)['payment_value'].sum()

,payment_type,payment_value
0,boleto,2869361.27
1,credit_card,12542084.19
2,debit_card,217989.79
3,not_defined,0.00
4,voucher,379436.87


Q3.1 Turn the result of Q1.1 into a DataFrame with columns payment_type and total_payment, using reset_index. Confirm the type is DataFrame.

Ans. df = payments.groupby('payment_type')['payment_value'].sum().reset_index(name='total_payment')

Q3.2 Get the same table without reset_index, using as_index=False in the groupby. Do both give identical tables?

Ans. payments.groupby('payment_type', as_index=False)['payment_value'].sum()

The answer for Q3.1 without reset_index() generates a df where payment type in the index. then by using reset_index with parameter name, why is the name of the value column changing, shouldn't the name of index column change?

Also, what is the difference in Q3.1 & Q3.2? Both generate similar results except that with reset_index we can rename columns and with as_index we simply do not let the columns to be set as index.

Why name= 
renames the value column, not the index column

Look at what the groupby returns before reset_index:

python
s = payments.groupby("payment_type")["payment_value"].sum()

s is a Series. It has two parts, and each already has a name:

the index, named payment_type (taken from the groupby key),
the values, named payment_value (the Series' own name, taken from the column you summed).

reset_index() moves the index into a regular column, and it keeps the name the index already has, payment_type. There is nothing to rename there. The values column is the one that needs a name, and name="total_payment" sets it, replacing payment_value.

So the name= argument of Series.reset_index always refers to the values column, because a Series has only one column of values.

The index column only needs help if it has no name. In that case, it would show up as index. To rename the index column as well, do it before resetting:

python
s.rename_axis("pay_type").reset_index(name="total_payment")

======================================================================================

Q3.1 and Q3.2 compared

Your summary is almost right, with one correction. Both end up as a DataFrame with the same rows and the same numbers. They differ in when the flat table is made and in how you name the value column.

Q3.1 reset_index:
When - after the groupby result already exists, with the keys in the index
Value column name - total_payment (from name=)	payment_value (the original name)
Where - it works	after almost any operation: value_counts(), groupby, sort_values, a pivot.

Q3.2 as_index=False:
When - the keys never go into the index in the first place
Value column name - payment_value (the original name)
Where - only inside groupby

On renaming: 
the as_index=False version can name the output too, just not with a name= argument. Use named aggregation:

python
payments.groupby("payment_type", as_index=False).agg(total_payment=("payment_value", "sum"))

Or rename afterwards with .rename(columns={"payment_value": "total_payment"})

Which to use: as_index=False (or named agg) is tidier when you only need a groupby result. reset_index is more general, because you can use it on any Series or DataFrame whose index you want back as a column, and drop=True is available when you want to throw the index away instead.

## Task 4 - Build pivot_table and crosstab

### 4.3 Using pivot_table on payments, show for each payment_type the sum, mean and count of payment_value in one table.

When solving this question through a pivot table, we are not required to pass through the columns parameter. It is optional. Since pivot_table is a dataframe method that takes column name strings.

Whereas when using the crosstab method, we are required to pass through both the index and column parameters, since both are required, not optional.

Hence here, while using crosstab, we pass a string 'payment_value' that will act as a dummy column grouping all the aggregations.

Crosstab sees data as separate lists instead of one dataframe, hence requiring both the index and columns parameter. On the other hand, pivot_table works on a dataframe.

Rule of Thumb

• Use pivot_table() when your data is already inside a single DataFrame (95% of the time).
• Use crosstab() when you want to quickly count frequencies between two separate lists, or when an assignment specifically asks you to use it.


In [46]:
##Method-1
payments.pivot_table(index='payment_type',
                    aggfunc={"payment_value": ["sum","mean","count"]})

payment_value                         
                     count        mean          sum
payment_type                                       
boleto               19784  145.034435   2869361.27
credit_card          76795  163.319021  12542084.19
debit_card            1529  142.570170    217989.79
not_defined              3    0.000000         0.00
voucher               5775   65.703354    379436.87

In [28]:
##Method-2
pd.pivot_table(payments, index='payment_type', 
               values='payment_value', 
               aggfunc=['sum', 'mean', 'count'])

,sum,mean,count
,payment_value,payment_value,payment_value
payment_type,,,
boleto,2869361.27,145.034435,19784
credit_card,12542084.19,163.319021,76795
debit_card,217989.79,142.570170,1529
not_defined,0.00,0.000000,3
voucher,379436.87,65.703354,5775


In [45]:
pd.crosstab(values=payments.payment_value, index=payments.payment_type,
            columns='payment_value',
           aggfunc=['sum','mean','count'])

,sum,mean,count
col_0,payment_value,payment_value,payment_value
payment_type,,,
boleto,2869361.27,145.034435,19784
credit_card,12542084.19,163.319021,76795
debit_card,217989.79,142.570170,1529
not_defined,0.00,0.000000,3
voucher,379436.87,65.703354,5775


### 4.1 Build a crosstab of review_score (rows) against the review year (columns), using reviews.review_creation_date.str[:4] for the year.

In [30]:
pd.crosstab(index=reviews.review_score,columns=reviews.review_creation_date.str[:4])

review_creation_date,2016,2017,2018
review_score,,,
1,88,4597,6739
2,8,1350,1793
3,22,3589,4568
4,51,8440,10651
5,156,24759,32413


In [35]:
##Above question through pivot_table
reviews["year"] = reviews.review_creation_date.str[:4]
counts = reviews.pivot_table(values='review_id',
                    index='review_score',
                    columns='year',aggfunc='count')

### Q4.2 Show the same table as percentages within each year, so each column adds up to 100. Which year had the highest share of 1-star reviews?

In [36]:
pd.crosstab(reviews.review_score,reviews.review_creation_date.str[:4],
            normalize='columns').mul(100).round(2)

review_creation_date,2016,2017,2018
review_score,,,
1,27.08,10.76,12.00
2,2.46,3.16,3.19
3,6.77,8.40,8.13
4,15.69,19.75,18.96
5,48.00,57.94,57.71


In [48]:
##Above question through pivot table.
##Since pivot table has no normalize parameter, we need to convert counts to percentages in a second step.

pct = counts.div(counts.sum(axis=0), axis=1).mul(100).round(2)
pct

year,2016,2017,2018
review_score,,,
1,27.08,10.76,12.00
2,2.46,3.16,3.19
3,6.77,8.40,8.13
4,15.69,19.75,18.96
5,48.00,57.94,57.71


Here is what happens at each step of your code:

1. counts.sum(axis=0) → Sum DOWN each column

Setting axis=0 tells pandas to collapse the rows vertically to find the total for each column.


2. counts.div(..., axis=1) → Divide ACROSS matching columns

Now you have a 2D table (counts) and a 1D list of totals (Mumbai: 100, Pune: 40).
Setting axis=1 tells pandas: "Match the column names across the top, and divide every row by that column's total.

In [41]:
#which year has the highest share of 1-star reviews
pct.loc[1].idxmax()

'2016'

In [39]:
pct.loc[1]

year
2016    27.08
2017    10.76
2018    12.00
Name: 1, dtype: float64

In [40]:
pct.iloc[0]

year
2016    27.08
2017    10.76
2018    12.00
Name: 1, dtype: float64

## Task 5 - Filter groups, the HAVING equivalent, with boolean indexing after agg

### 5.1 From items, find the product_id values that appear in more than 50 rows. How many are there?

In [49]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [77]:
##Method-1
p = items.groupby('product_id').agg(
    count = ('product_id', 'size')
)
p[p['count'] > 50]
len(p)

32951

In [78]:
##Method-2
s = items.groupby('product_id').size()
s[s > 50]
len(s)

32951

### 5.2 Using named aggregation, find the product_id values with more than 50 rows and an average price above 100.

In [88]:
#Method-1
a = items.groupby('product_id').agg(
    count = ('product_id','count'),
    avg_price = ('price','mean') 
)
a[(a['count'] > 50) & (a['avg_price'] > 100)]

,count,avg_price
product_id,,
0a57f7d2c983bcf8188589a5fea4a8da,104,114.492308
165f86fe8b799a708a20ee4ba125c289,109,163.494587
19c91ef95d509ea33eda93495c4d3481,156,121.086154
24c66f106f642621e524291a895c9032,65,181.900000
25f86162fee18735fffdb762dcb10d7c,62,153.609677
...,...,...
f4f67ccaece962d013a4e1d7dc3a61f7,56,234.639286
f71973c922ccaab05514a36a8bc741b8,65,102.886000
fbc1488c1a1e72ba175f53ab29a248e8,58,118.491207


In [87]:
#Method-2
g = items.groupby('product_id').agg(n=('price', 'size'), avg=('price', 'mean'))
g[(g.n > 50) & (g.avg > 100)]

,n,avg
product_id,,
0a57f7d2c983bcf8188589a5fea4a8da,104,114.492308
165f86fe8b799a708a20ee4ba125c289,109,163.494587
19c91ef95d509ea33eda93495c4d3481,156,121.086154
24c66f106f642621e524291a895c9032,65,181.900000
25f86162fee18735fffdb762dcb10d7c,62,153.609677
...,...,...
f4f67ccaece962d013a4e1d7dc3a61f7,56,234.639286
f71973c922ccaab05514a36a8bc741b8,65,102.886000
fbc1488c1a1e72ba175f53ab29a248e8,58,118.491207


### 5.3 Get the original item rows (not summary rows) for products with more than 50 rows, using groupby().filter(). How is the result different from Q5.1?

Unlike .agg() or .apply() (which usually summarize data into a single number per product), .filter() returns the original, un-summarized rows—it simply removes the groups that don't pass your True/False test.

In [89]:
items.groupby('product_id').filter(lambda x : len(x) > 50)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
10,00061f2a7bc09da83e415a52dc8a4af1,1,d63c1011f49d98b976c352955b1c4bea,cc419e0650a3c5ba77189a1882b7556a,2018-03-29 22:28:09,59.99,8.88
12,0006ec9db01a64e59a68b2c340bf65a7,1,99a4788cb24856965c36a24e339b6058,4a3ca9315b744ce9f8e9374361493884,2018-07-26 17:24:20,74.00,23.32
13,0008288aa423d2a3f00fcb17cd7d8719,1,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.90,13.37
14,0008288aa423d2a3f00fcb17cd7d8719,2,368c6c730842d78016ad823897a372db,1f50f920176fa81dab994f9023523100,2018-02-21 02:55:52,49.90,13.37
28,00130c0eee84a3d909e75bc08c5c3ca1,1,89321f94e35fc6d7903d36f74e351d40,16090f2ca825584b5a147ab24aa30c86,2018-06-14 05:16:24,27.90,7.94
...,...,...,...,...,...,...,...
112636,fff90cdcb3b2e6cfb397d05d562fd3fe,1,764292b2b0f73f77a0272be03fdd45f3,bd23da7354813347129d751591d1a6e2,2017-11-30 10:11:28,89.90,11.83
112640,fffb9224b6fc7c43ebb0904318b10b5f,1,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112641,fffb9224b6fc7c43ebb0904318b10b5f,2,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19
112642,fffb9224b6fc7c43ebb0904318b10b5f,3,43423cdffde7fda63d0414ed38c11a73,b1fc4f64df5a0e8b6913ab38803c57a9,2017-11-03 02:55:58,55.00,34.19


## Exercise - From order_items, compute total revenue (sum of price), item count, and average price for each product_id. Keep only products with more than 100 items sold and show the top 10 by revenue.

In [91]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [95]:
a = items.groupby('product_id').agg(
    total_revenue = ('price','sum'),
    item_count = ('price','size'),
    avg_price = ('price','mean')
)
a[a['item_count'] > 100].sort_values(by='total_revenue', ascending=False).head(10)

,total_revenue,item_count,avg_price
product_id,,,
bb50f2e236e5eea0100680137654686c,63885.00,195,327.615385
6cdd53843498f92890544667809f1595,54730.20,156,350.834615
d1c427060a0f73f6b889a5c7c61f2ac4,47214.51,343,137.651633
99a4788cb24856965c36a24e339b6058,43025.56,488,88.167131
3dd2a17168ec895c781a9191c1e95ad7,41082.60,274,149.936496
53b36df67ebb7c41585e8d54d6772e08,37683.42,323,116.666935
aca2eb7d00ea1a7b8ebd4e68314663af,37608.90,527,71.364137
e0d64dcfaa3b6db5c54ca298ae101d05,31786.82,194,163.849588
d285360f29ac7fd97640bf0baef03de0,31623.81,123,257.104146
